In [5]:
!pip install qdrant-client
!pip install cohere langchain tfds-nightly langchain_community
!pip install langchain-openai langchain-cohere langchain-qdrant langchain-community langchain-text-splitters qdrant-client ipython

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 73.7 MB/s eta 0:00:00
  Attempting uninstall: cohere
    Found existing installation: cohere 7.2.0
    Uninstalling cohere-7.2.0:
      Successfully uninstalled cohere-7.2.0


In [16]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.5/395.5 kB 9.9 MB/s eta 0:00:00


In [ ]:
import cohere
co = cohere.Client("API_KEY")

In [14]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_cohere import CohereEmbeddings, ChatCohere
from langchain_qdrant import QdrantVectorStore
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.chains import RetrievalQA
from langchain_classic.chains import RetrievalQAWithSourcesChain
from langchain_classic.chains.question_answering import load_qa_chain
from langchain_core.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate,
    PromptTemplate,
)
from IPython.display import display, Markdown, Latex
import textwrap as tr
import random
import os

# Populate Text Embeddings into a VectorDB

In [ ]:
#save the embeddings in a DB that is persisten
from langchain_text_splitters import CharacterTextSplitter
loader = PyPDFLoader("./Introduction_to_RAG.pdf")
document=loader.load()
docs=document[0].page_content
text_splitter = CharacterTextSplitter(
  #Set a really small chunk size, just to show.
  separator ='\n\n',
  chunk_size = 100,
  chunk_overlap = 0,
  length_function = len,
  add_start_index = True,
)
docs = text_splitter.create_documents([docs])
texts = [doc.page_content for doc in docs]
#os.environ["COHERE_API_KEY"] 'u6N4NZ5vo9fPG1aAMGY@GG5ErYwOLPbp3J9Z0W3K'

#Define the embeddings model
embeddings = CohereEmbeddings (model = 'embed-multilingual-v3.0', cohere_api_key="API_KEY")
metadata=[{"source": text} for text in texts]

#Embed the documents and store in index
vector_store = QdrantVectorStore.from_texts(texts, embeddings, location=":memory:",metadatas=metadata, collection_name="summaries", distance_func="Dot")

# Query the VectorDB return Generation and Citations

In [ ]:
retriever = vector_store.as_retriever()

prompt_template="""You are Delores, an AI customer service agent that talks to guests staying in an AirBnB in the canary islands.
Answer the questions using the facts provided. You always uses pleasantries and polite greetings like "Good morning" and
"Good afternoon". Use the following pieces of context to answer the users question.
Take note of the sources and include them in the answer in the format: "SOURCES: source1 source2", use "SOURCES" in capital letters regardless of the number of sources.
If you don't know the answer, just say that "I don't know", don't try to make up an answer.
---------------
{summaries}"""
messages = [
  SystemMessagePromptTemplate.from_template(prompt_template),
  HumanMessagePromptTemplate.from_template("{question}")
]
prompt = ChatPromptTemplate.from_messages (messages)

chain_type_kwargs = {"prompt": prompt}

llm = ChatCohere(model="command-nightly", cohere_api_key="API_KEY")

chain = RetrievalQAWithSourcesChain.from_chain_type(
  llm=llm,
  chain_type="stuff",
  retriever=vector_store.as_retriever(),
  # return_source_document=True,
  chain_type_kwargs=chain_type_kwargs
)

def print_result(result):
  output_text = f"""### Question:
  {query}
  ### Answer:
  {result['answer']}
  ### Source:
  {result['sources']}
  ### All relevant sources:
  """
  # {' '.join(list(set([doc.metadata['source'] for doc in result['source_documents']])))}

  return(output_text)

In [28]:
query = "what is RAG?"
result = chain(query)
display(Markdown(print_result(result)))

{'question': 'what is RAG?', 'answer': "Good morning! RAG stands for **Retrieval-Augmented Generation**, a smart approach designed to enhance the performance of **Large Language Models (LLMs)** like OpenAI's GPT-4, Google's Gemini, Anthropic Claude, and others. RAG improves how these models respond to queries, especially when the required information was not part of their training data. It does this by enabling LLMs to access and utilize **external information sources** while generating answers to user queries.\n\nRAG is particularly useful because early language models (pre-2020), such as GPT-2, BERT, and T5, had limitations: they couldn't access external knowledge at inference time, often hallucinated (made up facts), and couldn't stay up-to-date after training. RAG addresses these issues by combining **retrieval** (fetching relevant information from external sources) with **generation** (creating coherent and accurate responses).\n\nLeading LLMs using RAG include:\n- OpenAI GPT-4 wi

### Question:
  what is RAG?
  ### Answer:
  Good morning! RAG stands for **Retrieval-Augmented Generation**, a smart approach designed to enhance the performance of **Large Language Models (LLMs)** like OpenAI's GPT-4, Google's Gemini, Anthropic Claude, and others. RAG improves how these models respond to queries, especially when the required information was not part of their training data. It does this by enabling LLMs to access and utilize **external information sources** while generating answers to user queries.

RAG is particularly useful because early language models (pre-2020), such as GPT-2, BERT, and T5, had limitations: they couldn't access external knowledge at inference time, often hallucinated (made up facts), and couldn't stay up-to-date after training. RAG addresses these issues by combining **retrieval** (fetching relevant information from external sources) with **generation** (creating coherent and accurate responses).

Leading LLMs using RAG include:
- OpenAI GPT-4 with Browsing/Plugins
- Anthropic Claude with RAG pipelines
- Meta's RAG Model
- Cohere Command-R and Command-R+
- Llama 2/Llama 3 with LangChain or LlamaIndex
- Microsoft Azure OpenAI + Cognitive Search
- Google Gemini with enterprise connectors


  ### Source:
  Introduction to RAG
  ### All relevant sources:
  